# Data Preprocessing — Cáceres Solar Forecast (TFT)

This notebook transforms the assembled dataset (`data/caceres_assembled_dataset.csv`) into
train / validation / test splits ready for the Temporal Fusion Transformer.

**Key MLVU principle:** split first, compute all statistics on the training set only, then
apply to validation and test sets. This prevents data leakage and simulates a realistic
production scenario where the model has never seen future data.

**Pipeline steps:**

| # | Step | Principle |
|---|---|---|
| 0 | Load, verify temporal continuity, rename target | Sanity check |
| 1 | Temporal train / val / test split | Split before any transforms |
| 2 | Impute missing target with 0 | Domain-knowledge correction (overnight hours) |
| 3 | Clip physical artifacts | Physics constraint (precip/irradiance ≥ 0) |
| 4 | Remove night-time target outliers | Unphysical measurement error |
| 5 | Feature engineering | kt, dewpoint depression, day-of-year cyclical |
| 6 | Drop redundant features | Multicollinearity (clearsky_dni, clearsky_dhi) |
| 7 | Standardization (z-score) | Train-set mean/std applied to all splits |
| 8 | Linear time index | Capacity trend proxy |
| 9 | Feature classification for TFT | Known future vs. observed inputs |
| 10 | Save outputs | CSVs + preprocessing_params.json |

In [1]:
import pandas as pd
import numpy as np
import json, os

DATA_DIR = os.path.join(os.getcwd(), 'data')
RAW_CSV  = os.path.join(DATA_DIR, 'caceres_assembled_dataset.csv')

## Step 0 — Load, verify temporal continuity, rename target

- Confirm 26,304 hourly rows with no gaps (2022-01-01 00:00 → 2024-12-31 23:00)
- Rename `pv_generation_gwh` → `pv_generation_mwh` (confirmed unit is MWh, not GWh)

In [ ]:
df = pd.read_csv(RAW_CSV, parse_dates=['datetime_utc'], index_col='datetime_utc')

# verify temporal continuity
expected = pd.date_range('2023-01-01', '2025-12-31 23:00:00', freq='h')
assert len(df) == len(expected), f"Row count mismatch: {len(df)} vs {len(expected)}"
assert df.index.equals(expected), "Timestamp gaps detected"

# rename target column — unit is MWh, not GWh
df = df.rename(columns={'pv_generation_gwh': 'pv_generation_mwh'})
TARGET = 'pv_generation_mwh'

print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"Date range: {df.index[0]} → {df.index[-1]}")
print(f"Target column: {TARGET}")
print(f"Missing target values: {df[TARGET].isna().sum():,} ({df[TARGET].isna().mean():.1%})")
df.head(3)

## Step 1 — Temporal train / validation / test split

Split **before** computing any statistics. Cutoffs chosen for clean calendar boundaries:

| Set | Date Range | Seasons covered |
|---|---|---|
| Train | 2022-01-01 → 2023-12-31 | 2 full calendar years (all seasons × 2) |
| Val | 2024-01-01 → 2024-06-30 | Winter → Summer |
| Test | 2024-07-01 → 2024-12-31 | Summer → Winter (highest capacity, closest to production) |

In [ ]:
TRAIN_END = '2024-12-31 23:00:00'
VAL_END   = '2025-06-30 23:00:00'

train = df.loc[:'2024-12-31 23:00:00'].copy()
val   = df.loc['2025-01-01':VAL_END].copy()
test  = df.loc['2025-07-01':].copy()

# verify no overlap and full coverage
assert len(train) + len(val) + len(test) == len(df), "Split doesn't cover all rows"
assert train.index[-1] < val.index[0], "Train/val overlap"
assert val.index[-1] < test.index[0], "Val/test overlap"

splits = {'train': train, 'val': val, 'test': test}
for name, s in splits.items():
    pct = len(s) / len(df) * 100
    missing_tgt = s[TARGET].isna().sum()
    print(f"{name:5s}: {len(s):6,} rows ({pct:5.1f}%)  |  {s.index[0]} → {s.index[-1]}  |  missing target: {missing_tgt:,}")

## Step 2 — Impute missing target with 0

The EDA showed that 100% of missing target values occur overnight (solar zenith > 90°, consecutive
gaps of 9–13 hours matching winter night duration). ESIOS simply doesn't report near-zero generation.
This is a **domain-knowledge correction**, not a statistical estimate — safe to apply uniformly to all
splits without data leakage concerns.

In [4]:
for name, s in splits.items():
    n_before = s[TARGET].isna().sum()
    s[TARGET] = s[TARGET].fillna(0.0)
    print(f"{name:5s}: imputed {n_before:,} missing values → 0.0")

# sanity: no NaN should remain anywhere
for name, s in splits.items():
    assert s.isna().sum().sum() == 0, f"NaN values remain in {name}"
print("\nNo NaN values in any split")

train: imputed 2,921 missing values → 0.0
val  : imputed 1,024 missing values → 0.0
test : imputed 857 missing values → 0.0

No NaN values in any split


## Step 3 — Clip physical artifacts

ERA5 reanalysis produces tiny negative values for `total_precip_mm` (min ≈ −2.8e-5) and
`ssrd_wm2` (min ≈ −5.7e-4) due to floating-point arithmetic. Both quantities are physically
non-negative. Clip to 0 — this is a physics constraint, not a learned transform.

In [5]:
clip_cols = ['total_precip_mm', 'ssrd_wm2']
for name, s in splits.items():
    for col in clip_cols:
        n_neg = (s[col] < 0).sum()
        s[col] = s[col].clip(lower=0.0)
        if n_neg > 0:
            print(f"{name:5s}: clipped {n_neg} negative values in {col}")

print("Physical clipping done")

train: clipped 360 negative values in total_precip_mm
train: clipped 123 negative values in ssrd_wm2
val  : clipped 66 negative values in total_precip_mm
val  : clipped 40 negative values in ssrd_wm2
test : clipped 69 negative values in total_precip_mm
test : clipped 38 negative values in ssrd_wm2
Physical clipping done


## Step 4 — Remove night-time target outliers

The EDA found a maximum night-time generation of 52 MWh when `solar_zenith > 95°`. For a
~2.5 GWp fleet, this is unphysical — solar panels produce no meaningful energy when the sun
is well below the horizon. The MLVU lecture labels such points "unnatural outliers"
(measurement/data errors) as opposed to natural extremes.

In [6]:
NIGHT_ZENITH = 95       # degrees — sun well below horizon
NIGHT_GEN_THRESHOLD = 0.1  # MWh — conservative; small residuals kept

for name, s in splits.items():
    night_outlier = (s['solar_zenith'] > NIGHT_ZENITH) & (s[TARGET] > NIGHT_GEN_THRESHOLD)
    n_outliers = night_outlier.sum()
    if n_outliers > 0:
        max_val = s.loc[night_outlier, TARGET].max()
        s.loc[night_outlier, TARGET] = 0.0
        print(f"{name:5s}: zeroed {n_outliers} night-time outliers (max was {max_val:.1f} MWh)")
    else:
        print(f"{name:5s}: no night-time outliers found")

print("Night-time outlier removal done")

train: zeroed 1134 night-time outliers (max was 52.0 MWh)
val  : zeroed 202 night-time outliers (max was 43.6 MWh)
test : zeroed 313 night-time outliers (max was 44.3 MWh)
Night-time outlier removal done


## Step 5 — Feature engineering

All new features are deterministic functions of existing columns — no training statistics needed.

| Feature | Formula | Rationale |
|---|---|---|
| `kt` | `ssrd_wm2 / clearsky_ghi` (clipped [0, 1.5]) | Decouples cloud attenuation from solar geometry |
| `dewpoint_depression_C` | `temperature_2m_C − dewpoint_2m_C` | Humidity proxy; raw dewpoint had r = 0.016 with target |
| `doy_sin`, `doy_cos` | sin/cos of day-of-year / 365.25 | Finer seasonal signal than month_sin/cos |

In [7]:
for name, s in splits.items():
    # --- clear-sky index (kt) ---
    # only compute when clearsky_ghi > 1 W/m2 to avoid division noise near sunrise/sunset
    s['kt'] = np.where(
        s['clearsky_ghi'] > 1.0,
        s['ssrd_wm2'] / s['clearsky_ghi'],
        0.0
    )
    s['kt'] = s['kt'].clip(0.0, 1.5)

    # --- dewpoint depression (humidity proxy) ---
    s['dewpoint_depression_C'] = s['temperature_2m_C'] - s['dewpoint_2m_C']

    # --- day-of-year cyclical encoding ---
    doy = s.index.dayofyear
    s['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
    s['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)

# quick check on kt distribution (training set, daytime only)
daytime_kt = train.loc[train['clearsky_ghi'] > 1.0, 'kt']
print(f"kt (train, daytime): mean={daytime_kt.mean():.3f}, median={daytime_kt.median():.3f}, "
      f"std={daytime_kt.std():.3f}, max={daytime_kt.max():.3f}")
print(f"dewpoint_depression_C (train): min={train['dewpoint_depression_C'].min():.2f}, "
      f"max={train['dewpoint_depression_C'].max():.2f}")
print(f"\nNew columns added: kt, dewpoint_depression_C, doy_sin, doy_cos")
print(f"Train shape: {train.shape}")

kt (train, daytime): mean=0.859, median=0.883, std=0.359, max=1.500
dewpoint_depression_C (train): min=0.06, max=38.93

New columns added: kt, dewpoint_depression_C, doy_sin, doy_cos
Train shape: (17520, 20)


## Step 6 — Drop redundant features

`clearsky_dni` and `clearsky_dhi` are highly collinear with `clearsky_ghi` (r > 0.93). All three
come from the same pvlib Ineichen model. Keeping `clearsky_ghi` alone (used as kt denominator)
is sufficient. PCA was considered but rejected — dropping 2 explicit features is simpler and
preserves interpretability, and the TFT has built-in variable selection.

In [8]:
drop_cols = ['clearsky_dni', 'clearsky_dhi']
for name, s in splits.items():
    splits[name] = s.drop(columns=drop_cols)

# reassign for convenience
train, val, test = splits['train'], splits['val'], splits['test']

print(f"Dropped: {drop_cols}")
print(f"Remaining columns ({train.shape[1]}): {list(train.columns)}")

Dropped: ['clearsky_dni', 'clearsky_dhi']
Remaining columns (18): ['dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa', 'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'pv_generation_mwh', 'kt', 'dewpoint_depression_C', 'doy_sin', 'doy_cos']


## Step 7 — Standardization (z-score)

Compute mean and standard deviation on the **training set only**, then apply to all splits.

- **Standardize:** all continuous weather, solar, and derived features + the target
- **Do NOT standardize:** cyclical sin/cos features (already in [−1, 1] by construction;
  shifting their center would break sinusoidal semantics)

In [9]:
# features to z-score standardize
features_to_scale = [
    'dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2',
    'solar_zenith', 'solar_azimuth', 'clearsky_ghi',
    'kt', 'dewpoint_depression_C',
]
# cyclical features — NOT scaled
cyclical_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']

# compute stats from training set ONLY
train_means = train[features_to_scale].mean()
train_stds  = train[features_to_scale].std()
target_mean = train[TARGET].mean()
target_std  = train[TARGET].std()

print("Training set statistics for standardization:")
print(pd.DataFrame({'mean': train_means, 'std': train_stds}).round(4))
print(f"\nTarget ({TARGET}): mean={target_mean:.4f}, std={target_std:.4f}")

Training set statistics for standardization:
                           mean       std
dewpoint_2m_C            8.3278    4.5918
temperature_2m_C        17.4867    8.4251
surface_pressure_hPa   973.2410    5.7201
total_precip_mm          0.0699    0.2814
ssrd_wm2               201.9422  282.4840
strd_wm2               324.9577   41.7274
solar_zenith            89.6658   36.3020
solar_azimuth          181.1067  100.6959
clearsky_ghi           229.7436  306.9422
kt                       0.4257    0.4982
dewpoint_depression_C    9.1589    7.6129

Target (pv_generation_mwh): mean=460.5729, std=617.9445


In [10]:
# apply standardization to all splits
for name in splits:
    splits[name][features_to_scale] = (splits[name][features_to_scale] - train_means) / train_stds
    splits[name][TARGET] = (splits[name][TARGET] - target_mean) / target_std

train, val, test = splits['train'], splits['val'], splits['test']

# verify: training set should have mean ~ 0, std ~ 1 for scaled features
train_check = train[features_to_scale].agg(['mean', 'std']).T
print("Post-standardization check (train set):")
print(train_check.round(6))
print(f"\nTarget — train mean: {train[TARGET].mean():.6f}, std: {train[TARGET].std():.6f}")

Post-standardization check (train set):
                       mean  std
dewpoint_2m_C          -0.0  1.0
temperature_2m_C       -0.0  1.0
surface_pressure_hPa   -0.0  1.0
total_precip_mm         0.0  1.0
ssrd_wm2                0.0  1.0
strd_wm2                0.0  1.0
solar_zenith           -0.0  1.0
solar_azimuth          -0.0  1.0
clearsky_ghi            0.0  1.0
kt                     -0.0  1.0
dewpoint_depression_C   0.0  1.0

Target — train mean: -0.000000, std: 1.000000


## Step 8 — Linear time index (capacity trend proxy)

Installed capacity grew 2022→2024 (visible as increasing generation in the EDA). Without
per-month capacity data, we add a linear `time_idx` as a known future input, min-max normalized
using the training range. Values > 1.0 in val/test are intentional — they signal to the model
that capacity continues to increase beyond what it saw during training.

In [ ]:
epoch = pd.Timestamp('2023-01-01')

for name in splits:
    splits[name]['time_idx'] = (splits[name].index - epoch).total_seconds()

# min-max normalize using TRAINING range
time_min = splits['train']['time_idx'].min()
time_max = splits['train']['time_idx'].max()

for name in splits:
    splits[name]['time_idx'] = (splits[name]['time_idx'] - time_min) / (time_max - time_min)

train, val, test = splits['train'], splits['val'], splits['test']

print(f"time_idx range — train: [{train['time_idx'].min():.4f}, {train['time_idx'].max():.4f}]")
print(f"time_idx range — val:   [{val['time_idx'].min():.4f}, {val['time_idx'].max():.4f}]")
print(f"time_idx range — test:  [{test['time_idx'].min():.4f}, {test['time_idx'].max():.4f}]")

## Step 9 — Feature classification for TFT

The Temporal Fusion Transformer distinguishes between:
- **Known future inputs:** deterministic at forecast time (calendar, solar geometry, time index)
- **Observed inputs:** only available for past timesteps (weather measurements, derived weather features)
- **Target:** the variable to forecast

In [12]:
known_future = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'solar_zenith', 'solar_azimuth', 'clearsky_ghi', 'time_idx',
]
observed = [
    'dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2',
    'kt', 'dewpoint_depression_C',
]
target_col = [TARGET]

# verify all columns accounted for
all_classified = set(known_future + observed + target_col)
all_columns = set(train.columns)
assert all_classified == all_columns, f"Mismatch: {all_columns - all_classified} unclassified, {all_classified - all_columns} extra"

print(f"Known future inputs ({len(known_future)}): {known_future}")
print(f"Observed inputs    ({len(observed)}):      {observed}")
print(f"Target             (1):                    {target_col}")
print(f"Total: {len(all_classified)} columns")

Known future inputs (10): ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi', 'time_idx']
Observed inputs    (8):      ['dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa', 'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'kt', 'dewpoint_depression_C']
Target             (1):                    ['pv_generation_mwh']
Total: 19 columns


## Step 10 — Save outputs

Save:
1. **Three CSV files** — one per split, with consistent column ordering
2. **`preprocessing_params.json`** — all parameters needed to reproduce the transforms and
   denormalize predictions at inference time

In [ ]:
# consistent column order: known future, observed, target
col_order = known_future + observed + target_col

for name in splits:
    out_path = os.path.join(DATA_DIR, f'{name}_processed.csv')
    splits[name][col_order].to_csv(out_path)
    print(f"Saved {out_path}  ({splits[name].shape[0]:,} rows x {splits[name].shape[1]} cols)")

# save preprocessing parameters
params = {
    'split_boundaries': {
        'train': ['2023-01-01 00:00:00', TRAIN_END],
        'val':   ['2025-01-01 00:00:00', VAL_END],
        'test':  ['2025-07-01 00:00:00', '2025-12-31 23:00:00'],
    },
    'target_column': TARGET,
    'features_to_scale': features_to_scale,
    'cyclical_features': cyclical_features,
    'train_means': train_means.to_dict(),
    'train_stds': train_stds.to_dict(),
    'target_mean': float(target_mean),
    'target_std': float(target_std),
    'time_idx_min': float(time_min),
    'time_idx_max': float(time_max),
    'time_idx_epoch': '2023-01-01',
    'dropped_columns': drop_cols,
    'feature_classification': {
        'known_future': known_future,
        'observed': observed,
        'target': target_col,
    },
    'night_outlier_params': {
        'zenith_threshold': NIGHT_ZENITH,
        'generation_threshold_mwh': NIGHT_GEN_THRESHOLD,
    },
}

params_path = os.path.join(DATA_DIR, 'preprocessing_params.json')
with open(params_path, 'w') as f:
    json.dump(params, f, indent=2)
print(f"\nSaved {params_path}")

## Verification

In [14]:
print("=" * 70)
print("VERIFICATION CHECKS")
print("=" * 70)

# 1. no NaN in any split
for name in splits:
    nans = splits[name].isna().sum().sum()
    assert nans == 0, f"{name} has {nans} NaN values"
print("[PASS] No NaN values in any split")

# 2. contiguous, non-overlapping dates
assert splits['train'].index[-1] < splits['val'].index[0]
assert splits['val'].index[-1] < splits['test'].index[0]
total_rows = sum(len(s) for s in splits.values())
assert total_rows == 26_304
print("[PASS] Splits are contiguous, non-overlapping, cover full dataset")

# 3. training set standardization
for col in features_to_scale:
    m = splits['train'][col].mean()
    s = splits['train'][col].std()
    assert abs(m) < 1e-6, f"{col} mean = {m}"
    assert abs(s - 1.0) < 1e-6, f"{col} std = {s}"
t_mean = splits['train'][TARGET].mean()
t_std = splits['train'][TARGET].std()
assert abs(t_mean) < 1e-6 and abs(t_std - 1.0) < 1e-6
print("[PASS] Standardized features have mean ~ 0, std ~ 1 on training set")

# 4. cyclical features untouched
for col in cyclical_features:
    assert splits['train'][col].min() >= -1.001
    assert splits['train'][col].max() <= 1.001
print("[PASS] Cyclical features remain in [-1, 1]")

# 5. overnight target spot-check
expected_zero_zscore = (0.0 - target_mean) / target_std
most_common = splits['train'][TARGET].mode().iloc[0]
print(f"[INFO] Most common target value (z-scored): {most_common:.6f}")
print(f"[INFO] Expected z-score of 0 MWh: {expected_zero_zscore:.6f}")

# 6. kt z-score of 0 at night
kt_zero_zscore = -train_means['kt'] / train_stds['kt']
print(f"[INFO] Expected z-score of kt=0: {kt_zero_zscore:.6f}")

# 7. summary
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
for name in ['train', 'val', 'test']:
    s = splits[name]
    print(f"{name:5s}: {s.shape[0]:6,} rows x {s.shape[1]} cols  |  "
          f"{s.index[0].strftime('%Y-%m-%d')} -> {s.index[-1].strftime('%Y-%m-%d')}")
print(f"\nFeatures: {len(known_future)} known future + {len(observed)} observed + 1 target = {len(all_classified)} total")
print("All checks passed")

VERIFICATION CHECKS
[PASS] No NaN values in any split
[PASS] Splits are contiguous, non-overlapping, cover full dataset
[PASS] Standardized features have mean ~ 0, std ~ 1 on training set
[PASS] Cyclical features remain in [-1, 1]
[INFO] Most common target value (z-scored): -0.745330
[INFO] Expected z-score of 0 MWh: -0.745330
[INFO] Expected z-score of kt=0: -0.854460

FINAL SUMMARY
train: 17,520 rows x 19 cols  |  2022-01-01 -> 2023-12-31
val  :  4,368 rows x 19 cols  |  2024-01-01 -> 2024-06-30
test :  4,416 rows x 19 cols  |  2024-07-01 -> 2024-12-31

Features: 10 known future + 8 observed + 1 target = 19 total
All checks passed
